In [1]:
# integrated_qstream_cq_demo.py

import math
import hashlib
import random
from dataclasses import dataclass
from typing import List, Tuple

from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit import transpile

# ============================================================
# 1. Quantum RNG (same structure as your notebook)
# ============================================================

def qrng_bits(num_bytes: int) -> bytes:
    num_bits = num_bytes * 8
    qc = QuantumCircuit(1)
    qc.h(0)
    qc.measure_all()

    backend = Aer.get_backend("qasm_simulator")
    tqc = transpile(qc, backend)
    counts = backend.run(tqc, shots=num_bits).result().get_counts()

    bits = []
    for bitval, count in counts.items():
        bits.extend(bitval * count)
    bits = bits[:num_bits]

    out = bytearray()
    for i in range(0, num_bits, 8):
        chunk = bits[i:i+8]
        val = 0
        for b in chunk:
            val = (val << 1) | (1 if b == '1' else 0)
        out.append(val)
    return bytes(out)

# ============================================================
# 2. Hash / XOR helpers + KDF with shared secret (Variation 1)
#    (secret only known to endpoints, not to q-stream service)
# ============================================================

def h( bytes) -> bytes:
    return hashlib.sha256(data).digest()

def xor_bytes(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))

def kdf_from_kgen_and_secret(kgen_blocks: List[bytes],
    secret: bytes,
    key_bytes: int) -> bytes:
    """
    Simple HKDF-like construction:
    master = H(secret || concat(kGen_i)) iterated until key_bytes reached.
    """
    master = b""
    blocks_concat = b"".join(kgen_blocks)
    counter = 0
    while len(master) < key_bytes:
        counter += 1
        material = secret + blocks_concat + counter.to_bytes(4, "big")
        master += h(material)
    return master[:key_bytes]

# ============================================================
# 3. q-stream: UserState, registration, server, extraction
#    (adapted from your notebook, with lastLM handling)
# ============================================================

@dataclass
class UserState:
    name: str
    otc: bytes
    U: bytes
    counter: int = 0
    last_LM: bytes = b""

    def next_ruid(self) -> bytes:
        self.counter += 1
        data = self.otc + self.U + self.counter.to_bytes(8, "big") + self.last_LM
        return h(data)

def register_user(name: str, identity: str) -> UserState:
    otc = qrng_bits(32)            # 256-bit OTC
    U = identity.encode("utf-8")   # user-specific data (e.g., email, hull ID)
    return UserState(name=name, otc=otc, U=U)

@dataclass
class QStreamServer:
    block_bits: int = 2_097_152
    num_blocks: int = 18
    min_kgen_bits: int = 128
    max_kgen_bits: int = 256

    def generate_R_and_metadata(
        self,
        users: List[UserState],
    ) -> Tuple[bytes, dict]:
        block_bytes = self.block_bits // 8
        R = bytearray(qrng_bits(block_bytes))
        meta = {u.name: [] for u in users}

        cursor = 0
        for ordinal in range(self.num_blocks):
            length_bits = self.min_kgen_bits + (
                qrng_bits(2)[0] % (self.max_kgen_bits - self.min_kgen_bits + 1)
            )
            length_bytes = math.ceil(length_bits / 8)
            if cursor + length_bytes >= block_bytes:
                cursor = 0

            start = cursor
            cursor += length_bytes

            for u in users:
                ruid = u.next_ruid()
                P = (
                    start.to_bytes(4, "big")
                    + length_bytes.to_bytes(2, "big")
                    + ordinal.to_bytes(1, "big")
                )
                S = ruid[: len(P)]
                PS = xor_bytes(P, S)

                offset = qrng_bits(1)[0] % 32
                pos = min(start + offset, block_bytes - len(ruid) - len(PS))

                R[pos:pos + len(ruid)] = ruid
                R[pos + len(ruid):pos + len(ruid) + len(PS)] = PS

                meta[u.name].append((start, length_bytes, ordinal))
        return bytes(R), meta

def extract_kgen_for_user(
    R: bytes,
    user: UserState,
    num_blocks: int = 18,
) -> List[bytes]:
    kgen_blocks = []
    search_from = 0
    for _ in range(num_blocks):
        ruid = user.next_ruid()
        idx = R.find(ruid, search_from)
        if idx == -1:
            break
        ps_start = idx + len(ruid)
        ps_end = ps_start + 7
        PS = R[ps_start:ps_end]
        S = ruid[: len(PS)]
        P = xor_bytes(PS, S)

        start = int.from_bytes(P[0:4], "big")
        length = int.from_bytes(P[4:6], "big")
        ordinal = P[6]

        kgen = R[start:start + length]
        kgen_blocks.append((ordinal, kgen))

        user.last_LM = kgen
        search_from = ps_end

    kgen_blocks.sort(key=lambda x: x[0])
    return [blk for _, blk in kgen_blocks]

# ============================================================
# 4. Simple Cq-style channel model with and without control
#    (bit-flip noise channel; "control" reduces error rate)
# ============================================================

@dataclass
class CQChannel:
    base_flip_prob: float = 0.05     # noise-only flip prob
    control_strength: float = 0.5    # 0=no control, 1=full mitigation

    def _effective_flip_prob(self) -> float:
        """
        Toy model for integrated non-commuting control:
        control reduces effective error rate.
        """
        p = self.base_flip_prob * (1.0 - 0.8 * self.control_strength)
        return max(0.0, min(0.5, p))

    def transmit(self, bits: bytes) -> bytes:
        """
        View 'bits' as classical symbols carried by underlying EM quantum states.
        Channel acts as a binary symmetric channel with effective p.
        """
        p_eff = self._effective_flip_prob()
        out = bytearray()
        for b in bits:
            flipped = 0
            for i in range(8):
                bit = (b >> i) & 1
                if random.random() < p_eff:
                    bit ^= 1
                flipped |= (bit << i)
            out.append(flipped)
        return bytes(out)

# ============================================================
# 5. Naval link demo: Ship A ↔ Ship B via q-stream over Cq channel
# ============================================================

def otp(key: bytes, msg: bytes) -> bytes:
    return xor_bytes(key, msg)

def naval_link_demo():
    random.seed(42)  # for reproducibility of the classical noise

    # --- Parties (ships) register with q-stream ---
    alice = register_user("Ship-A", "shipA@navy.example")
    bob   = register_user("Ship-B", "shipB@navy.example")

    # Shared secret only between A and B (Variation 1 in the paper)
    shared_secret_AB = b"CLASSIFIED-SEA-LINK-SECRET"

    server = QStreamServer()

    # --- q-stream service generates a QRNG block R and embeds rUIDs ---
    R_raw, meta = server.generate_R_and_metadata([alice, bob])

    # --- R is sent over a Cq-optimized physical link ---

    # Channel without control (baseline)
    chan_no_control = CQChannel(base_flip_prob=0.05, control_strength=0.0)
    # Channel with integrated control (reduced effective p)
    chan_with_control = CQChannel(base_flip_prob=0.05, control_strength=0.8)

    # Choose which we want to use:
    #   R_over_link = chan_no_control.transmit(R_raw)
    R_over_link = chan_with_control.transmit(R_raw)

    # --- Each ship extracts its kGen from the (possibly noisy) R ---

    # For a realistic simulation, if noise is high, extraction may fail or yield
    # mismatched kGen. Here we just run the algorithm and check equality.

    alice_kgen = extract_kgen_for_user(R_over_link, alice)
    bob_kgen   = extract_kgen_for_user(R_over_link, bob)

    # If noise is too high, they may not match; for now, we just check.
    if not alice_kgen or not bob_kgen:
        print("Extraction failed due to channel noise; reduce base_flip_prob or increase control.")
        return

    # --- Derive a shared session key with KDF + secret ---
    key_bytes = 64
    KA = kdf_from_kgen_and_secret(alice_kgen, shared_secret_AB, key_bytes)
    KB = kdf_from_kgen_and_secret(bob_kgen,   shared_secret_AB, key_bytes)

    if KA != KB:
        print("Key mismatch (channel errors too high or model too simple).")
        return

    # --- Use key as OTP for a naval message ---
    msg = b"Launch authorization window: 12:00-12:15Z, sector ALPHA."
    key_for_msg = KA[:len(msg)]
    ct = otp(key_for_msg, msg)

    # Ship B decrypts:
    key_for_msg_B = KB[:len(msg)]
    pt = otp(key_for_msg_B, ct)

    print("Cq channel effective flip probability:",
        chan_with_control._effective_flip_prob())
    print("Ciphertext (hex):", ct.hex())
    print("Recovered plaintext:", pt.decode("utf-8", errors="replace"))

if __name__ == "__main__":
    naval_link_demo()

NameError: name 'data' is not defined

In [ ]:
pip install qiskit qiskit-aer qiskit-ibm-runtime qiskit-ibm-provider
pip install qiskit[visualization]